## PostgreSQL Schema

Generate the PostgreSQL database schema and save it for backend handoff.

In [0]:
# Define the output location for handoff files.

handoff_dir = "/Volumes/risknet_catalog/risknet/risknet_volume/handoff"
schema_path = f"{handoff_dir}/schema.sql"

In [0]:
# Store the complete PostgreSQL schema as a SQL string.

schema_sql = """
---------------------Table 1: applicant---------------------
CREATE TABLE applicant (
    applicant_id UUID PRIMARY KEY,
    applicant_external_id VARCHAR(50) UNIQUE,
    name VARCHAR(100) NOT NULL,
    email VARCHAR(150) UNIQUE,
    phone VARCHAR(20),
    date_of_birth DATE,
    customer_age INTEGER,
    employment_status VARCHAR(30),
    housing_status VARCHAR(30),
    income NUMERIC(12,2),
    credit_risk_score NUMERIC(5,2),
    account_created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);


---------------------  Table 2: alert -------------------------

CREATE TABLE alert (
    alert_id UUID PRIMARY KEY,
    applicant_id UUID NOT NULL,
    risk_score NUMERIC(5,2) NOT NULL,
    risk_level VARCHAR(20) NOT NULL,
    alert_type VARCHAR(50),
    is_false_positive BOOLEAN DEFAULT FALSE,
    status VARCHAR(20) DEFAULT 'OPEN',
    generated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT fk_alert_applicant
        FOREIGN KEY (applicant_id)
        REFERENCES applicant(applicant_id)
        ON DELETE CASCADE
);


-----------------------Table 3: device------------------------

CREATE TABLE device (
    device_id UUID PRIMARY KEY,
    device_fingerprint VARCHAR(255) UNIQUE NOT NULL,
    device_type VARCHAR(50),
    operating_system VARCHAR(50),
    browser VARCHAR(50),
    first_seen TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    last_seen TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);



----------------------- Table 4: ip_address -----------------------

CREATE TABLE ip_address (
    ip_id UUID PRIMARY KEY,
    ip_address VARCHAR(45) UNIQUE NOT NULL,
    ip_version VARCHAR(10),
    country VARCHAR(100),
    region VARCHAR(100),
    city VARCHAR(100),
    is_vpn BOOLEAN DEFAULT FALSE,
    first_seen TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    last_seen TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);




------------------------Table 5: investigation_case---------------------

CREATE TABLE investigation_case (
    case_id UUID PRIMARY KEY,
    alert_id UUID NOT NULL,
    assigned_analyst VARCHAR(100),
    priority VARCHAR(20) DEFAULT 'MEDIUM',
    status VARCHAR(20) DEFAULT 'OPEN',
    opened_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    closed_at TIMESTAMP,
    remarks TEXT,

    CONSTRAINT fk_case_alert
        FOREIGN KEY (alert_id)
        REFERENCES alert(alert_id)
        ON DELETE CASCADE
);





------------------------ Table 6: analyst_decision  -----------------------

CREATE TABLE analyst_decision (
    decision_id UUID PRIMARY KEY,
    case_id UUID NOT NULL,
    analyst_name VARCHAR(100) NOT NULL,
    decision VARCHAR(20) NOT NULL,
    decision_reason TEXT,
    decided_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT fk_decision_case
        FOREIGN KEY (case_id)
        REFERENCES investigation_case(case_id)
        ON DELETE CASCADE
);
  



------------------------ Table 7: audit_log  -----------------------
   
CREATE TABLE audit_log (
    audit_id UUID PRIMARY KEY,
    applicant_id UUID,
    alert_id UUID,
    case_id UUID,
    decision_id UUID,
    action VARCHAR(100) NOT NULL,
    performed_by VARCHAR(100) NOT NULL,
    action_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    details TEXT,

    CONSTRAINT fk_audit_applicant
        FOREIGN KEY (applicant_id)
        REFERENCES applicant(applicant_id)
        ON DELETE SET NULL,

    CONSTRAINT fk_audit_alert
        FOREIGN KEY (alert_id)
        REFERENCES alert(alert_id)
        ON DELETE SET NULL,

    CONSTRAINT fk_audit_case
        FOREIGN KEY (case_id)
        REFERENCES investigation_case(case_id)
        ON DELETE SET NULL,

    CONSTRAINT fk_audit_decision
        FOREIGN KEY (decision_id)
        REFERENCES analyst_decision(decision_id)
        ON DELETE SET NULL
);




------------------------ Table 8:applicant_device -----------------------

CREATE TABLE applicant_device (
    applicant_id UUID NOT NULL,
    device_id UUID NOT NULL,

    PRIMARY KEY (applicant_id, device_id),

    CONSTRAINT fk_applicant_device_applicant
        FOREIGN KEY (applicant_id)
        REFERENCES applicant(applicant_id)
        ON DELETE CASCADE,

    CONSTRAINT fk_applicant_device_device
        FOREIGN KEY (device_id)
        REFERENCES device(device_id)
        ON DELETE CASCADE
);




------------------------ Table 9: applicant_ip  -----------------------

CREATE TABLE applicant_ip (
    applicant_id UUID NOT NULL,
    ip_id UUID NOT NULL,

    PRIMARY KEY (applicant_id, ip_id),

    CONSTRAINT fk_applicant_ip_applicant
        FOREIGN KEY (applicant_id)
        REFERENCES applicant(applicant_id)
        ON DELETE CASCADE,

    CONSTRAINT fk_applicant_ip_ip
        FOREIGN KEY (ip_id)
        REFERENCES ip_address(ip_id)
        ON DELETE CASCADE
);

"""

In [0]:
# Save the schema SQL file to the Unity Catalog Volume.

with open(schema_path, "w") as file:
    file.write(schema_sql)

print(f"Schema saved successfully: {schema_path}")

Schema saved successfully: /Volumes/risknet_catalog/risknet/risknet_volume/handoff/schema.sql


## Neo4j Cypher Import Commands

Generate Cypher scripts to recreate the graph from exported CSV files.

In [0]:
# Define the output path for the Neo4j Cypher script.

cypher_path = "/Volumes/risknet_catalog/risknet/risknet_volume/handoff/neo4j_import.cypher"

In [0]:
import pandas as pd

graph_export = "/Volumes/risknet_catalog/risknet/risknet_volume/graph_export"

folders = [
    "applicant_nodes",
    "device_nodes",
    "ip_nodes",
    "alert_nodes",
    "applicant_device_relationships",
    "applicant_ip_relationships",
    "alert_applicant_relationships"
]

for folder in folders:
    print(f"\n{folder}")
    df = spark.read.option("header", True).csv(
        f"/Volumes/risknet_catalog/risknet/risknet_volume/graph_export/{folder}"
    )
    print(df.columns)


applicant_nodes
['ApplicantID', 'credit_risk_score', 'employment_status', 'housing_status', 'customer_age', 'income']

device_nodes
['DeviceID']

ip_nodes
['IPAddress']

alert_nodes
['AlertID', 'RegistrationTimestamp', 'is_false_positive_alert']

applicant_device_relationships
['ApplicantID', 'DeviceID']

applicant_ip_relationships
['ApplicantID', 'IPAddress']

alert_applicant_relationships
['AlertID', 'ApplicantID']


In [0]:
# Store the Neo4j import commands as a Cypher script.

cypher_script = """
// ---------- Applicant Nodes ----------
LOAD CSV WITH HEADERS FROM 'file:///applicant_nodes.csv' AS row
CREATE (:Applicant {
    ApplicantID: row.ApplicantID,
    credit_risk_score: toFloat(row.credit_risk_score),
    employment_status: row.employment_status,
    housing_status: row.housing_status,
    customer_age: toInteger(row.customer_age),
    income: toFloat(row.income)
});

// ---------- Device Nodes ----------
LOAD CSV WITH HEADERS FROM 'file:///device_nodes.csv' AS row
CREATE (:Device {
    DeviceID: row.DeviceID
});

// ---------- IP Nodes ----------
LOAD CSV WITH HEADERS FROM 'file:///ip_nodes.csv' AS row
CREATE (:IPAddress {
    IPAddress: row.IPAddress
});

// ---------- Alert Nodes ----------
LOAD CSV WITH HEADERS FROM 'file:///alert_nodes.csv' AS row
CREATE (:Alert {
    AlertID: row.AlertID,
    RegistrationTimestamp: row.RegistrationTimestamp,
    is_false_positive_alert: row.is_false_positive_alert
});

// ---------- Indexes ----------
CREATE INDEX FOR (a:Applicant) ON (a.ApplicantID);
CREATE INDEX FOR (d:Device) ON (d.DeviceID);
CREATE INDEX FOR (i:IPAddress) ON (i.IPAddress);
CREATE INDEX FOR (al:Alert) ON (al.AlertID);

// ---------- Applicant -> Device ----------
LOAD CSV WITH HEADERS FROM 'file:///applicant_device_relationships.csv' AS row
MATCH (a:Applicant {ApplicantID: row.ApplicantID})
MATCH (d:Device {DeviceID: row.DeviceID})
CREATE (a)-[:USES_DEVICE]->(d);

// ---------- Applicant -> IP ----------
LOAD CSV WITH HEADERS FROM 'file:///applicant_ip_relationships.csv' AS row
MATCH (a:Applicant {ApplicantID: row.ApplicantID})
MATCH (i:IPAddress {IPAddress: row.IPAddress})
CREATE (a)-[:USES_IP]->(i);

// ---------- Alert -> Applicant ----------
LOAD CSV WITH HEADERS FROM 'file:///alert_applicant_relationships.csv' AS row
MATCH (al:Alert {AlertID: row.AlertID})
MATCH (a:Applicant {ApplicantID: row.ApplicantID})
CREATE (al)-[:GENERATED_FOR]->(a);
"""

In [0]:
# Save the Cypher import script to the Unity Catalog Volume.

with open(cypher_path, "w") as file:
    file.write(cypher_script)

print(f"Neo4j import script saved: {cypher_path}")

Neo4j import script saved: /Volumes/risknet_catalog/risknet/risknet_volume/handoff/neo4j_import.cypher


## Sample JSON for Frontend

Generate a sample API response for frontend development and testing.

In [0]:
# Define the output location for the sample frontend JSON.

json_path = "/Volumes/risknet_catalog/risknet/risknet_volume/handoff/sample_response.json"

In [0]:
# Create a sample API response matching the frontend contract.

import json

sample_response = {
    "applicantId": "APP00122916",
    "riskScore": 82.7,
    "riskLevel": "HIGH",
    "isFalsePositive": False,
    "deviceId": "LDEV298523",
    "ipAddress": "172.16.48.103",
    "registrationTimestamp": "2025-03-18T14:26:35Z",
    "graphSummary": {
        "sharedDevices": 4,
        "sharedIPs": 3,
        "connectedApplicants": 9
    },
    "explanations": [
        {
            "feature": "Shared Device Count",
            "impact": 0.42
        },
        {
            "feature": "Credit Risk Score",
            "impact": 0.27
        },
        {
            "feature": "Income",
            "impact": -0.11
        }
    ]
}

In [0]:
# Save the sample API response to the Unity Catalog Volume.

with open(json_path, "w") as file:
    json.dump(sample_response, file, indent=4)

print(f"Sample JSON saved: {json_path}")

Sample JSON saved: /Volumes/risknet_catalog/risknet/risknet_volume/handoff/sample_response.json


## Dataset Summary

Generate a summary of the processed dataset and exported graph components for project documentation.

In [0]:
# Define the output path.

summary_path = "/Volumes/risknet_catalog/risknet/risknet_volume/handoff/dataset_summary.txt"

In [0]:
# Read the exported graph data and generate a dataset summary.
graph_export_path='/Volumes/risknet_catalog/risknet/risknet_volume/graph_export/'

applicant_count = spark.read.option("header", True).csv(
    f"{graph_export_path}/applicant_nodes"
).count()

device_count = spark.read.option("header", True).csv(
    f"{graph_export_path}/device_nodes"
).count()

ip_count = spark.read.option("header", True).csv(
    f"{graph_export_path}/ip_nodes"
).count()

alert_count = spark.read.option("header", True).csv(
    f"{graph_export_path}/alert_nodes"
).count()

applicant_device_count = spark.read.option("header", True).csv(
    f"{graph_export_path}/applicant_device_relationships"
).count()

applicant_ip_count = spark.read.option("header", True).csv(
    f"{graph_export_path}/applicant_ip_relationships"
).count()

alert_applicant_count = spark.read.option("header", True).csv(
    f"{graph_export_path}/alert_applicant_relationships"
).count()

summary = f"""
===========================
RiskNet Dataset Summary
===========================

Applicants                    : {applicant_count}
Devices                       : {device_count}
IP Addresses                  : {ip_count}
Alerts                        : {alert_count}

Applicant-Device Relationships: {applicant_device_count}
Applicant-IP Relationships    : {applicant_ip_count}
Alert-Applicant Relationships : {alert_applicant_count}

Export Location:
/Volumes/risknet_catalog/risknet/risknet_volume/graph_export

Generated Files:
- PostgreSQL Schema (schema.sql)
- Neo4j Import Script (neo4j_import.cypher)
- Sample Frontend JSON (sample_response.json)
- Dataset Summary (dataset_summary.txt)
"""

In [0]:
# Save the dataset summary.

with open(summary_path, "w") as file:
    file.write(summary)

print(summary)
print(f"\nDataset summary saved: {summary_path}")


RiskNet Dataset Summary

Applicants                    : 1000000
Devices                       : 938296
IP Addresses                  : 942832
Alerts                        : 109714

Applicant-Device Relationships: 1000000
Applicant-IP Relationships    : 1000000
Alert-Applicant Relationships : 109714

Export Location:
/Volumes/risknet_catalog/risknet/risknet_volume/graph_export

Generated Files:
- PostgreSQL Schema (schema.sql)
- Neo4j Import Script (neo4j_import.cypher)
- Sample Frontend JSON (sample_response.json)
- Dataset Summary (dataset_summary.txt)


Dataset summary saved: /Volumes/risknet_catalog/risknet/risknet_volume/handoff/dataset_summary.txt


## Handoff Summary

The following artifacts have been generated for deployment and integration with the backend and frontend components of RiskNet.

In [0]:
print("""
==========================================
        RiskNet Handoff Complete
==========================================

Generated Artifacts

1. PostgreSQL Schema
   schema.sql

2. Neo4j Import Script
   neo4j_import.cypher

3. Sample Frontend Response
   sample_response.json

4. Dataset Summary
   dataset_summary.txt

5. Graph Export Directory
   applicant_nodes/
   device_nodes/
   ip_nodes/
   alert_nodes/
   applicant_device_relationships/
   applicant_ip_relationships/
   alert_applicant_relationships/

Location:
/Volumes/risknet_catalog/risknet/risknet_volume/handoff

Status: Ready for Backend, Neo4j and Frontend integration.
""")


        RiskNet Handoff Complete

Generated Artifacts

1. PostgreSQL Schema
   schema.sql

2. Neo4j Import Script
   neo4j_import.cypher

3. Sample Frontend Response
   sample_response.json

4. Dataset Summary
   dataset_summary.txt

5. Graph Export Directory
   applicant_nodes/
   device_nodes/
   ip_nodes/
   alert_nodes/
   applicant_device_relationships/
   applicant_ip_relationships/
   alert_applicant_relationships/

Location:
/Volumes/risknet_catalog/risknet/risknet_volume/handoff

Status: Ready for Backend, Neo4j and Frontend integration.

